### setup

In [ ]:
import gzip
import os
from pathlib import Path
import tempfile
import warnings

import torch

import imfusion
import imfusion.machinelearning as ml
from imfusion import labels
import numpy as np

np.set_printoptions(precision=2, suppress=True)
warnings.filterwarnings("ignore", category=DeprecationWarning)

os.environ["__NV_PRIME_RENDER_OFFLOAD"] = "1"
os.environ["__GLX_VENDOR_LIBRARY_NAME"] = "nvidia"
os.environ["__VK_LAYER_NV_optimus"] = "NVIDIA_only"

# Downloads the images from TCIA on the first run, see data/brats-africa/README.md
%run data/brats-africa/fetch.py
BRATS_AFRICA_PROJECT_PATH = "data/brats-africa/project"


def predict(case_id: str) -> torch.Tensor:
    """Stand-in for a trained model: returns the saved prediction of a case, [1, 1, D, H, W]."""
    return torch.load(gzip.open(f"data/brats-africa/predictions/{case_id}.pt.gz"))

# From Data to Models

The ML lifecycle in the ImFusion ecosystem

MICCAI 2026 workshop · September 2026

Ilja Manakov · ImFusion GmbH

<div class="im-agenda"></div>

## What does it take to get from data to a model?

1. **organize** your data. Images, annotations and metadata in one place.
2. **build** a data pipeline. Directly from your Labels project.
3. **review** model predictions. Back in the project, next to the GT.
4. **deploy** your model. One configuration file for Python and C++.

---

<div class="im-dark"></div>

## 00

setup

*Up and running in three steps*

<div class="im-dark"></div>

## Getting started

Install the SDK from our package index. Its base modules include Labels and machine learning:

```sh
pip install "imfusion-sdk[base]" --extra-index-url https://pypi.imfusion.com/simple
```

Activate your license once, before the first import:

```python
import os
os.environ["IMFUSION_LICENSE_KEY"] = "<your-license-key>"

import imfusion
```

The ImFusion Labels application is available as an [installer](https://downloads.imfusion.com/Webshop/new_webshop_labels.html).

<p class="im-footnote"><a href="https://docs.imfusion.com/python/imfusion-sdk/1.0.0/getting_started/installing.html">Installation guide</a></p>

- `imfusion-sdk[base]` includes `imfusion-sdk-labels` and `imfusion-sdk-machinelearning`
- Modality-specific modules are extras on top of it, e.g. `[spine]`, `[ultrasound]`, `[computed-tomography]`, `[vision]`

---

<div class="im-blue"></div>

## 01

organize

*Know your data before you train on it.*

## Case study: BraTS-Africa 2024

Pre-operative brain MRI of patients from Nigeria, with expert tumor annotations.

**146** patients · **584** MR volumes · **4** sequences · **5** centers · **4** scanner models · glioma and other neoplasms

Keeping it all organized throughout an ML project is a challenge.

<p class="im-footnote"><a href="https://doi.org/10.7937/v8h6-8x67">Adewole et al., BraTS-Africa (2024), The Cancer Imaging Archive</a> · CC BY 4.0</p>

<p><img class="im-contain" src="pictures/brats_case.png" alt=""></p>

---

<div class="im-dark"></div>

## ImFusion Labels

Live demo

In [ ]:
%%capture
%%bash
ImFusionLabels open-project data/brats-africa/project

- Open `data/brats-africa/project` in Labels
- Database view: expand a patient -> T1, T1c, T2, FLAIR
- Image metadata columns: dimensions, spacing, modality, source path
- Filter `Sequence = T2`, then `Set = Training`
- Group by TumorType, Center or ScannerInformation
- Usual route: export the data from Labels and train on the export, Labels supports this
- Section 02 shows the Labels loader instead, still experimental: it trains on the living project, a view of it, not on exports that go stale, get lost and are hard to keep track of
- The same filter comes back in Python in section 02

---

<div class="im-blue"></div>

## 02

build

*From Labels project to training batches in a few lines.*

## Get it from the source!

Data exports eventually become stale and are cumbersome to manage.
`imfusion.machinelearning.Dataset` can read images and annotation layers straight from a Labels project.

We can specify
- a `layers` configuration that selects which annotations should be loaded
- a `tags` configurations that selects which tags should be loaded
- a `filter` configuration that filters the database via tags

### We configure the data loader with a Properties object

In [ ]:
reader_config = imfusion.Properties(dict(path=BRATS_AFRICA_PROJECT_PATH, data_name="t2"))
reader_config.add_sub_properties("layers").add_sub_properties("GT")["field_name"] = "segmentation"
reader_config.add_sub_properties("tags").add_sub_properties("TumorType")["field_name"] = "classification"
reader_config.add_sub_properties("filter").copy_from(dict(Sequence="T2", Set="Training"))
dataset = ml.Dataset("labels_project", reader_config)
print(f"Dataset has {len(dataset)} samples.")

### Let's load and inspect a sample

In [ ]:
sample = next(iter(dataset))
imfusion.show(sample)
print(sample)

## Inject custom logic at any point

A pipeline is only as flexible as its escape hatch. `map` is ours.

Here we use it to add the other three sequences of each patient.

You can use it for anything specific to your data, cases no framework could anticipate.

### imfusion.labels allows us to interact with the project directly

In [ ]:
# Construct a look-up table from T2 descriptor to the remaining sequences
project = labels.Project.load(BRATS_AFRICA_PROJECT_PATH)
group_to_desc = {}
for d in project.descriptors:
    group_to_desc.setdefault(d.grouping[0], {})[d.tags["Sequence"].value] = d
seq_to_rest = {d.name: group_to_desc[d.grouping[0]] for d in project.descriptors}


SEQUENCES = ["FLAIR", "T1", "T1c", "T2"]
def add_sequences(item):
    sequences = seq_to_rest[item["t2"].sis.name]
    for sequence in SEQUENCES[:-1]:
        item[sequence.lower()] = sequences[sequence].load_image(crop_to_roi=False)


dataset.map(add_sequences)

- The image keeps the name of its descriptor (`sis.name`), that is our entry point
- The project groups descriptors by Patient (`project.grouping_hierarchy`), `d.grouping` is the patient ID
- We find the other sequences by patient and `Sequence` tag, no file name conventions involved

## Batteries included!

`imfusion.machinelearning` offers a wide assortment of processing operations:

In [ ]:
ml.OperationsSequence.available_operations()

### Build the processing pipeline one operation at a time

In [ ]:
FIELDS = [s.lower() for s in SEQUENCES]
dataset.preprocess([
    ml.MarkAsTargetOperation(apply_to=["segmentation", "classification"]),
    ml.MakeFloatOperation(),
    ml.NormalizePercentileOperation(0.01, 0.99, ignore_zeros=True, clamp_values=True),
    ml.MergeAsChannelsOperation(apply_to=FIELDS, output_field="image"),
])

- Normalization is per sequence; `clamp_values` keeps the background at 0 and the range at [0, 1]
- `MergeAsChannels` keeps the `apply_to` order: FLAIR, T1, T1c, T2
- Note: Operations have sensible defaults for whether they are applied to inputs, targets or both. Since we mark "segmentation" and "classification" as targets, we don't have to single them out in the other ops.

## Don't repeat what is already done

`memory_cache` will create a snapshot in memory to avoid recomputing the same processing each epoch.

If you want something more persistent or have a large dataset that does not fit into memory, use `disk_cache` instead.

In [ ]:
dataset.memory_cache(lazy=True)

from time import perf_counter
for i in range(2):
    start = perf_counter()
    next(iter(dataset))
    print(f"Run {i} took {perf_counter() - start:.2f} seconds")
    next(iter(dataset))


- Run 0 loads the four sequences, normalizes, merges and fills the cache: talk over it
- Run 1 reads the same case from the cache
- A cached case is ~143 MB (4 float channels), all 116 training cases ~17 GB: `disk_cache()` or `memory_cache(compression=...)` for real training

## Only compute what changes

The `machinelearning` module offers many Operations for data augmentation and ROI sampling, which you can freely combine to fit your need.

### Combine multiple sampling strategies with defined probabilities

In [ ]:
roi = [96, 96, 96]
dataset.sample(
    [ml.RandomROISampler(roi),
     ml.LabelROISampler(roi, labels_values=[1, 2, 3])],
    weights=[0.67, 0.33]
)

### Use pre-defined RandomOperations and even group them via `ml.RandomChoiceOperation`

In [ ]:
dataset.preprocess([
    ml.RandomAxisFlipOperation(axes=["x"], probability=0.5),
    ml.RandomAddRandomNoiseOperation(type="gaussian", intensity_range=(0.01, 0.05), probability=0.25),
    ml.RandomChannelDropoutOperation(channel_drop_probability=0.25)
])

### Lastly, group your data and let it prefetch in the background

In [ ]:
dataset.batch(2)
dataset.prefetch(prefetch_size=6)

- Sampling comes before the augmentations, so these only process the 96³ patch
- `LabelROISampler` picks one of the label values present, then one of its voxels, and centers the ROI on it: small sub-regions are chosen as often as the edema
- Gaussian noise approximates MRI noise inside the brain; channel dropout simulates missing sequences

### Putting it all together

In [ ]:
dataset = (
    ml.Dataset("labels_project", reader_config)
    .map(add_sequences)
    .preprocess([
        ml.MarkAsTargetOperation(apply_to=["segmentation", "classification"]),
        ml.MakeFloatOperation(),
        ml.NormalizePercentileOperation(0.01, 0.99, ignore_zeros=True, clamp_values=True),
        ml.MergeAsChannelsOperation(apply_to=FIELDS, output_field="image"),
    ])
    .memory_cache(lazy=True)
    .sample(  
        [ml.RandomROISampler(roi), ml.LabelROISampler(roi, labels_values=[1, 2, 3])],
        weights=[0.67, 0.33]
    )
    .preprocess([
        ml.RandomAxisFlipOperation(axes=["x"], probability=0.5),
        ml.RandomAddRandomNoiseOperation(type="gaussian", intensity_range=(0.01, 0.05), probability=0.25),
        ml.RandomChannelDropoutOperation(channel_drop_probability=0.25)
    ])
    .batch(2)
    .prefetch(prefetch_size=6)
)

In [ ]:
item = next(iter(dataset)) 
imfusion.show(item)

## Traceability comes for free

Each image carries its dataset license and a record of every operation applied to it.

### Each image carries the dataset license stored in the Labels project

In [ ]:
item["image"].components.dataset_license

- Labels attaches the license and citation from the project settings to every image it loads
- Operations and batching keep it, the batch lists one entry per image

### The processing record lists every operation applied to the image

In [ ]:
item["image"].components.processing_record.configuration()

- Random operations record the parameters they drew, e.g. the noise intensity and the dropped channel
- `item["segmentation"]` lists the crop, the sampler and the flip, but none of the intensity augmentations

## And then hand off to pytorch

Converting the elements of a DataItem to PyTorch tensors is a one-liner

In [ ]:
x = item["image"].torch()
y = item["segmentation"].torch()

x.shape, y.shape

---

<div class="im-blue"></div>

## 03

review

*Model predictions, back in Labels.*

<div class="im-split-grid">
<div>
<h2>Understand the strengths and weaknesses of your model</h2>
<p>Aggregated metrics leave a dinosaur-sized hole in your understanding.</p>
<p>We need to go deeper.</p>
</div>
<div class="im-pics">
<div class="im-pic"><img src="pictures/datasaurus_dino.png" alt=""></div>
<div class="im-pic"><img src="pictures/datasaurus_star.png" alt=""></div>
<div class="im-pic"><img src="pictures/datasaurus_circle.png" alt=""></div>
<div class="im-pic"><img src="pictures/datasaurus_v_lines.png" alt=""></div>
</div>
</div>

<p class="im-footnote">Datasaurus Dozen: <a href="https://doi.org/10.1145/3025453.3025912">Matejka &amp; Fitzmaurice, Same Stats, Different Graphs, CHI 2017</a>, after Alberto Cairo</p>

- Four of the 13 Datasaurus Dozen datasets: dino, star, circle and vertical lines. All share the same means, standard deviations and correlation shown in the plots
- Any model works here: nnU-Net, MONAI or plain PyTorch, we only need one label map per case
- `predict` from the setup is a stand-in for the user's model: it returns saved outputs of the Cranial tumor model (section 04) as tensors

### Add a prediction layer and Dice tags

In [ ]:
DICE_TAGS = {1: "Dice_NCR", 2: "Dice_ED", 3: "Dice_ET"}

gt = next(layer for layer in project.settings().layers if layer.name == "GT")
prediction_layer = project.add_labelmap_layer("Prediction")
for c in [c for c in gt.labels if c.value]:
    prediction_layer.add_annotation(c.name, c.value, c.color)
    project.add_tag(DICE_TAGS[c.value], labels.TagKind.Float, c.color)

### Write each prediction and its Dice scores

In [ ]:
for d in project.descriptors:

    if d.tags["Sequence"].value != "T2" or d.tags["Set"].value != "Validation":
        continue

    gt = d.labelmap_layers["GT"].load()
    prediction = imfusion.SharedImageSet.from_torch(predict(d.grouping[0]), get_metadata_from=gt)
    dice = ml.DiceMetric().compute_dice(prediction, gt)[0]

    d.labelmap_layers["Prediction"].save_new_data(prediction)
    for category, tag in DICE_TAGS.items():
        if dice.get(category, None) is None:
            continue
        d.tags[tag] = float(dice[category])

project.save()

- The tensor holds only voxels, `get_metadata_from` copies spacing, orientation and modality from the GT
- Needs `project` from the lookup cell of section 02 and the cell above
- Reset the project with `git checkout data/brats-africa/project && git clean -fd data/brats-africa/project`

---

<div class="im-dark"></div>

## ImFusion Labels

Live demo 2, dicey boogaloo

In [ ]:
%%capture
%%bash
ImFusionLabels open-project data/brats-africa/project

- The launch cell opens the project with the new Prediction layer and the Dice tags, colored like their sub-regions
- Filter `Sequence = T2` and `Set = Validation`, group by TumorType, sort by Dice_NCR ascending
- Other neoplasms score much lower on the necrotic core: mean Dice_NCR 0.38 vs. 0.69 for glioma (ED and ET 0.66 vs. 0.79)
- Open one of them, toggle GT and Prediction
- A class missing from both GT and prediction scores 1, a class predicted but missing from GT scores 0

---

<div class="im-blue"></div>

## 04

deploy

*One configuration file per model.*

## Deploying a model with the ImFusion SDK

A traced network and one YAML configuration is all `MachineLearningModel` needs to run inference. The configuration describes the engine, preprocessing, patching and postprocessing.

`MachineLearningModel` is part of the C++ SDK, so the same configuration runs in Python and C++.

## The model configuration

```yaml
Engine:
  Name: torch
  ModelFile: MR-Heteromodal-TumorSegmentation.pt
PreProcessing:
  - NormalizeUniform: {}
  - Resample: {resolution: [1, 1, 1]}
  - SplitIntoPatches: {patch_size: [128, 128, 128]}
PostProcessing:
  - RecombinePatches: {mode: Weighted}
  - ResampleToInput: {}
  - ArgMax: {}
```

<p class="im-footnote">Excerpt from the Cranial tumor segmentation model configuration</p>

- The step names are the `ml` operations from section 02, without the `Operation` suffix
- The input is one image with FLAIR, T1, T1c, T2 as channels, the order of `SEQUENCES` in section 02

### We load the model from its configuration

In [ ]:
MODEL_YAML = "TumorSegmentation/MR-Heteromodal-TumorSegmentation-v2.yaml"
model = ml.MachineLearningModel(MODEL_YAML)

### The engine runs the network, on GPU or CPU

In [ ]:
engine = model.engine()
print("registered engines:", ml.Engine.available_engines())
print("providers:", engine.available_providers(), "| in use:", engine.provider())
print("force_cpu:", engine.force_cpu)

### Models with several inputs or outputs take a DataItem

In [ ]:
case_id = d.grouping[0]  # last validation case of section 03
images = [group_to_desc[case_id][s].load_image(crop_to_roi=False) for s in SEQUENCES]
merged, = imfusion.algorithm.execute("Base.MergeChannels", images)
item = ml.DataItem()
item["Input"] = merged
result = model.predict(item)

print(list(result.keys()), "| same as the saved prediction:",
      torch.equal(result["Prediction"].torch(), predict(case_id)))

- `model.predict` takes a single image as a shortcut, or a `DataItem` with one field per model input, the same type as in the section 02 pipeline
- The result is a `DataItem` with one field per model output, here `Prediction`
- The last line checks that the stand-in `predict` from the setup returns exactly this output
- One network run, about 5 s on the GPU and 20 s on the CPU
- `engine.force_cpu = True` when the GPU is busy; a failed prediction logs "Prediction failed" and returns None
- Other engines: ONNX Runtime, OpenVINO, CoreML; custom engines subclass `ml.Engine`

## Deploying your own model

1. **train** it, with nnU-Net, MONAI or plain PyTorch
2. **export** the network, e.g. as TorchScript or ONNX
3. **describe** it in a YAML configuration: engine, pre- and postprocessing
4. **run** it with `MachineLearningModel`, in Python or C++

Same pre- and postprocessing for training and deployment. Nothing to reimplement, nothing to keep in sync.

<div class="im-blue"></div>

# Thank you

<a href="https://docs.imfusion.com/python">docs.imfusion.com/python</a> · <a href="https://docs.imfusion.com/labels">docs.imfusion.com/labels</a>

manakov@imfusion.com